Importing cleaned dataset

In [1]:
import pandas as pd
df = pd.read_csv('cleaned_house_prices.csv')
df.info() #sanity check - should show 0 missing

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1459 entries, 0 to 1458
Data columns (total 77 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1459 non-null   int64  
 1   MSSubClass     1459 non-null   int64  
 2   MSZoning       1459 non-null   object 
 3   LotFrontage    1459 non-null   float64
 4   LotArea        1459 non-null   int64  
 5   Street         1459 non-null   object 
 6   LotShape       1459 non-null   object 
 7   LandContour    1459 non-null   object 
 8   Utilities      1459 non-null   object 
 9   LotConfig      1459 non-null   object 
 10  LandSlope      1459 non-null   object 
 11  Neighborhood   1459 non-null   object 
 12  Condition1     1459 non-null   object 
 13  Condition2     1459 non-null   object 
 14  BldgType       1459 non-null   object 
 15  HouseStyle     1459 non-null   object 
 16  OverallQual    1459 non-null   int64  
 17  OverallCond    1459 non-null   int64  
 18  YearBuil

Handpicking Features and handling textual data

In [3]:
# select the columns we're keeping
features = ['OverallQual', 'GrLivArea', 'GarageCars', 'TotalBsmtSF', 
            'FullBath', 'YearBuilt', 'Neighborhood', 'ExterQual', 'KitchenQual', 'SalePrice']
df_model = df[features].copy()

#now KitchenQual and ExterQual are text but have a real order (bad to excellent)

'''quality_map + .map() — this is called ordinal encoding, 
you're telling the model "Ex is better than Gd is better than TA," 
which is true and useful information'''

# ORDERED categorical -> map to numbers manually, since orders matter ('po'=1 .. 'Ex'=5)
quality_map = {'Po':1, 'Fa':2, 'TA':3, 'Gd':4,'Ex':5 }
df_model['ExterQual'] = df_model['ExterQual'].map(quality_map)
df_model['KitchenQual'] = df_model['KitchenQual'].map(quality_map)

# Neighbourhood is also text but has nno order  

'''pd.get_dummies() — this is one-hot encoding, 
it turns Neighborhood into many columns like Neighborhood_Downtown, 
Neighborhood_Suburb, each 0 or 1. This avoids implying a false order'''

'''drop_first=True — drops one category to avoid redundancy 
(a small technical detail called the "dummy variable trap"
 — if you know all-but-one neighborhoods are 0, the last one is implied, 
 so keeping it is redundant)'''

# UNORDERED categorical -> one-hot encoding create 0/1 column per neighbourhood
df_model = pd.get_dummies(df_model, columns = ['Neighborhood'], drop_first = True)

print(df_model.shape)
print(df_model.head())



(1459, 33)
   OverallQual  GrLivArea  GarageCars  TotalBsmtSF  FullBath  YearBuilt  \
0            7       1710           2          856         2       2003   
1            6       1262           2         1262         2       1976   
2            7       1786           2          920         2       2001   
3            7       1717           3          756         1       1915   
4            8       2198           3         1145         2       2000   

   ExterQual  KitchenQual  SalePrice  Neighborhood_Blueste  ...  \
0          4            4     208500                 False  ...   
1          3            3     181500                 False  ...   
2          4            4     223500                 False  ...   
3          3            4     140000                 False  ...   
4          4            4     250000                 False  ...   

   Neighborhood_NoRidge  Neighborhood_NridgHt  Neighborhood_OldTown  \
0                 False                 False                 Fa

Saving feature engineered data


In [4]:
df_model.to_csv('model_ready_house_prices.csv', index=False)